# Progrentis Agent + Nanbeige — Kaggle GPU notebook

This notebook runs **Nanbeige llama.cpp, the existing Progrentis Agent backend, and its headless Playwright browser in one Kaggle runtime**. The model endpoint is `http://127.0.0.1:8080/v1`; there is no Cloudflare tunnel and no public inference URL.

## Before running

1. Select a Kaggle GPU accelerator (the existing server notebook targets **T4 x2**) and enable Internet access.
2. Attach the same inputs used by `2_run_nanbeige_server.ipynb`:
   - Dataset `nanbeige-4-3-33b-gguf`, containing `Nanbeige_Nanbeige4.3-33B-Q4_K_M.gguf`.
   - Dataset `llama-cpp-bin`, containing a compatible `llama-server` and its shared libraries.
3. Run cells from top to bottom. If the project source is not attached as a Kaggle input, the setup cell shallow-clones the public GitHub `main` branch. Push the latest workspace code first, or attach a Kaggle dataset containing the project if you need changes that are not on GitHub yet.
4. Model generation and browser automation run in Kaggle, not on your laptop. Website access still requires Kaggle Internet access. The notebook interface uses widgets in the notebook output and calls the existing agent/browser modules directly (there is no second browser process or public app tunnel).

The notebook creates a random, in-memory key for the loopback-only model server. It never prints that key. Progrentis credentials are entered into password widgets and sent only to the same-origin browser login form, not to the LLM. Do not put credentials in prompts or notebook source.

If the model fails to load on your GPU, change `MODEL_CTX` to `16384` or `8192` and rerun from the server-start cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

MODEL_DATASET = 'nanbeige-4-3-33b-gguf'
MODEL_FILE = 'Nanbeige_Nanbeige4.3-33B-Q4_K_M.gguf'
LLAMA_DATASET = 'llama-cpp-bin'
MODEL_PORT = 8080
MODEL_ALIAS = 'nanbeige-4.3-33b'
MODEL_CTX = 32768
GPU_LAYERS = 999
KV_CACHE = 'q4_0'
SERVER_LOG = Path('/tmp/progrentis-llama-server.log')
WORKING = Path('/kaggle/working')


def find_kaggle_input(slug: str) -> Path:
    roots = [Path('/kaggle/input')]
    for root in roots:
        direct = root / slug
        if direct.exists():
            return direct
        datasets = root / 'datasets'
        if datasets.exists():
            matches = list(datasets.glob(f'*/{slug}'))
            if matches:
                return matches[0]
        if root.exists():
            matches = [p for p in root.iterdir() if p.is_dir() and p.name.lower() == slug.lower()]
            if matches:
                return matches[0]
    raise FileNotFoundError(f'Attach the Kaggle input dataset {slug!r} in the notebook Input panel.')


def show_gpu_status() -> None:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True, check=False
    )
    if result.returncode:
        raise RuntimeError('No NVIDIA GPU was detected. Select a Kaggle GPU accelerator.')
    print('GPU name, total memory, free memory:')
    print(result.stdout.strip())


show_gpu_status()
MODEL_DIR = find_kaggle_input(MODEL_DATASET)
LLAMA_DIR = find_kaggle_input(LLAMA_DATASET)
MODEL_PATH = MODEL_DIR / MODEL_FILE
if not MODEL_PATH.is_file():
    found = [p.name for p in MODEL_DIR.rglob('*.gguf')]
    raise FileNotFoundError(f'Model file not found: {MODEL_PATH}. GGUF files found: {found}')
if not (LLAMA_DIR / 'llama-server').is_file():
    raise FileNotFoundError(f'llama-server was not found in attached input {LLAMA_DIR}.')

# This key exists only in this kernel. It is not printed or stored in the notebook.
import secrets
LOCAL_API_KEY = secrets.token_urlsafe(32)
print(f'Model: {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / (1024**3):.1f} GiB)')
print(f'llama.cpp input: {LLAMA_DIR}')
print(f'Local model endpoint: http://127.0.0.1:{MODEL_PORT}/v1')
print(f'Requested context: {MODEL_CTX} tokens; KV cache: {KV_CACHE}')

## Prepare the existing Progrentis Agent source

The notebook prefers an attached Kaggle input containing `backend/main.py`. Otherwise it clones the repository. This cell does not read `.env.example` or `.env`.

In [ ]:
import shutil

input_root = Path('/kaggle/input')
attached_projects = [p for p in input_root.iterdir() if (p / 'backend' / 'main.py').is_file()] if input_root.exists() else []
if attached_projects:
    PROJECT_ROOT = attached_projects[0]
else:
    PROJECT_ROOT = WORKING / 'progrentis'
    if not (PROJECT_ROOT / 'backend' / 'main.py').is_file():
        if PROJECT_ROOT.exists():
            raise FileExistsError(
                f'{PROJECT_ROOT} exists but is not a Progrentis Agent checkout. '
                'Rename that specific directory yourself, or attach the project as a Kaggle input.'
            )
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', 'main',
             'https://github.com/Ender68-ai/progrentis.git', str(PROJECT_ROOT)],
            check=True
        )
if not (PROJECT_ROOT / 'backend' / 'main.py').is_file():
    raise FileNotFoundError('Project source must contain backend/main.py.')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt'), 'ipywidgets', 'requests'],
    check=True
)
subprocess.run(
    [sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'],
    check=True
)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Progrentis Agent source ready at {PROJECT_ROOT}')

## Prepare llama.cpp (local only)

Copies the attached server and shared libraries into writable Kaggle storage. No tunnel is installed.

In [ ]:
WORKING.mkdir(parents=True, exist_ok=True)
LLAMA_SERVER = WORKING / 'llama-server'
shutil.copy2(LLAMA_DIR / 'llama-server', LLAMA_SERVER)
LLAMA_SERVER.chmod(0o755)
for library in LLAMA_DIR.rglob('*.so*'):
    target = WORKING / library.name
    if library.is_file() and not target.exists():
        shutil.copy2(library, target)

os.environ['LD_LIBRARY_PATH'] = f"{WORKING}:{os.environ.get('LD_LIBRARY_PATH', '')}"
SERVER_ENV = os.environ.copy()
help_result = subprocess.run(
    [str(LLAMA_SERVER), '--help'], capture_output=True, text=True, env=SERVER_ENV
)
help_text = help_result.stdout + help_result.stderr
version_result = subprocess.run(
    [str(LLAMA_SERVER), '--version'], capture_output=True, text=True, env=SERVER_ENV
)
if version_result.returncode:
    raise RuntimeError('llama-server cannot start. Check that the attached binary matches the Kaggle GPU runtime and its shared libraries.')
if '--api-key' not in help_text or '--alias' not in help_text:
    raise RuntimeError('The attached llama-server must support --api-key and --alias; attach a compatible llama-cpp-bin dataset.')
print('llama.cpp binary verified:', (version_result.stdout or version_result.stderr).strip().splitlines()[0])
print('Server binds to loopback only; inference stays in this Kaggle runtime.')

## Start Nanbeige locally

If this cell is rerun, it stops only the server process started by this notebook. Model loading can take several minutes.

In [ ]:
import time
import requests
import signal

# Use the exact process handle started by this notebook; never kill by process name.
previous = globals().get('model_proc')
if previous is not None and previous.poll() is None:
    previous.terminate()
    try:
        previous.wait(timeout=15)
    except subprocess.TimeoutExpired:
        previous.kill()
        previous.wait(timeout=5)

required_options = [
    '--model', '--alias', '--host', '--port', '--n-gpu-layers', '--ctx-size',
    '--split-mode', '--cache-type-k', '--cache-type-v', '--api-key'
]
missing_options = [option for option in required_options if option not in help_text]
if missing_options:
    raise RuntimeError(f'llama-server binary is missing required options: {missing_options}')

os.environ['NO_PROXY'] = 'localhost,127.0.0.1'
os.environ['no_proxy'] = 'localhost,127.0.0.1'
SERVER_ENV = os.environ.copy()
cmd = [
    str(LLAMA_SERVER), '--model', str(MODEL_PATH), '--alias', MODEL_ALIAS,
    '--host', '127.0.0.1', '--port', str(MODEL_PORT),
    '--n-gpu-layers', str(GPU_LAYERS), '--ctx-size', str(MODEL_CTX),
    '--split-mode', 'layer', '--cache-type-k', KV_CACHE, '--cache-type-v', KV_CACHE,
    '--api-key', LOCAL_API_KEY, '--log-file', str(SERVER_LOG)
]
for option, value in [('--parallel', '1'), ('--temp', '0.2')]:
    if option in help_text:
        cmd.extend([option, value])
if '--flash-attn' in help_text:
    cmd.extend(['--flash-attn', 'on'])
model_proc = subprocess.Popen(
    cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=SERVER_ENV
)
headers = {'Authorization': f'Bearer {LOCAL_API_KEY}'}
ready = False
for elapsed in range(0, 901, 10):
    if model_proc.poll() is not None:
        break
    try:
        response = requests.get(f'http://127.0.0.1:{MODEL_PORT}/health', headers=headers, timeout=3)
        if response.ok:
            ready = True
            break
    except requests.RequestException:
        pass
    if elapsed % 30 == 0:
        print(f'Loading Nanbeige? {elapsed}s', flush=True)
    time.sleep(10)
if not ready:
    tail = SERVER_LOG.read_text(errors='replace').splitlines()[-60:] if SERVER_LOG.exists() else ['No llama-server log was created.']
    print('\n'.join(line.replace(LOCAL_API_KEY, '[redacted]') for line in tail))
    if model_proc.poll() is None:
        model_proc.terminate()
        try:
            model_proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            model_proc.kill()
            model_proc.wait(timeout=5)
    raise RuntimeError('Nanbeige did not become ready. For GPU out-of-memory, set MODEL_CTX=16384 or 8192 and rerun this cell.')

payload = {
    'model': MODEL_ALIAS,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: local model ready'}],
    'max_tokens': 24,
    'temperature': 0
}
check = requests.post(
    f'http://127.0.0.1:{MODEL_PORT}/v1/chat/completions',
    headers=headers, json=payload, timeout=3600
)
check.raise_for_status()
answer = check.json()['choices'][0]['message'].get('content', '')
print(f'Nanbeige is ready locally: {answer[:200]}')
print(f'Health endpoint: http://127.0.0.1:{MODEL_PORT}/health')

## Configure the app and open the notebook interface

The agent and Playwright browser run in this kernel. The widget UI replaces the external web page for this Kaggle workflow; it does not expose a server port.

In [ ]:
import asyncio
import html
import json
import ipywidgets as widgets
from IPython.display import display

os.environ.update({
    'LLM_BASE_URL': f'http://127.0.0.1:{MODEL_PORT}/v1',
    'LLM_API_KEY': LOCAL_API_KEY,
    'LLM_MODEL': MODEL_ALIAS,
    'LLM_TIMEOUT': '3600',
    'HOST': '127.0.0.1',
    'PORT': '8000',
    'LOG_LEVEL': 'INFO',
    'PROGRENTIS_USERNAME': '',
    'PROGRENTIS_PASSWORD': '',
    'NO_PROXY': 'localhost,127.0.0.1',
    'no_proxy': 'localhost,127.0.0.1',
})
from backend.config import settings
settings.LLM_BASE_URL = os.environ['LLM_BASE_URL']
settings.LLM_API_KEY = LOCAL_API_KEY
settings.LLM_MODEL = MODEL_ALIAS
settings.LLM_TIMEOUT = 3600
settings.HOST = '127.0.0.1'
settings.PORT = 8000

from backend.agent import run_agent
from backend.browser import browser

chat_log = widgets.Output(layout={'border': '1px solid #d5ddd3', 'height': '300px', 'overflow_y': 'auto', 'padding': '10px'})
page_view = widgets.HTML(value='<i>Open a page, then inspect it or ask about it.</i>')
status = widgets.HTML(value='<b>Model:</b> local ? <b>Browser:</b> idle')
prompt = widgets.Textarea(placeholder='Inspect this page and explain what the instructions mean?', layout=widgets.Layout(width='100%', height='80px'))
url = widgets.Text(placeholder='https://?', layout=widgets.Layout(width='75%'))
open_button = widgets.Button(description='Open URL', button_style='primary')
progrentis_button = widgets.Button(description='Open Progrentis sign-in')
inspect_button = widgets.Button(description='Inspect page')
send_button = widgets.Button(description='Ask agent', button_style='success')
stop_button = widgets.Button(description='Stop run', button_style='danger', disabled=True)
username = widgets.Text(description='Username:', layout=widgets.Layout(width='50%'))
password = widgets.Password(description='Password:', layout=widgets.Layout(width='50%'))
confirmed = widgets.Checkbox(value=False, description='I verified this is the official Progrentis sign-in page')
login_button = widgets.Button(description='Sign in to current page', button_style='warning')
shutdown_button = widgets.Button(description='Stop browser and model')
state = {'task': None}


def show_message(label: str, text: str, color: str = '#243b2c') -> None:
    with chat_log:
        display(widgets.HTML(
            f'<div style="margin:6px 0;padding:9px 12px;border-left:3px solid {color};background:#f4f6ef">'
            f'<b>{html.escape(label)}</b><br>{html.escape(text).replace(chr(10), "<br>")}</div>'
        ))


def show_observation(data: dict) -> None:
    safe = html.escape(json.dumps(data, ensure_ascii=False, indent=2))
    page_view.value = (
        f'<b>{html.escape(data.get("title") or "Untitled page")}</b><br>'
        f'<code>{html.escape(data.get("url") or "")}</code>'
        f'<pre style="max-height:320px;overflow:auto;white-space:pre-wrap">{safe}</pre>'
    )
    status.value = f'<b>Model:</b> local ? <b>Browser:</b> running ? <b>Page:</b> {html.escape(data.get("title") or "")}'


async def navigate_page(target_url: str):
    try:
        if not browser.is_running:
            await browser.start()
        observed = await browser.navigate(target_url.strip())
        show_observation(observed)
    except Exception as exc:
        show_message('Browser error', str(exc), '#b03a2e')


def on_open(_):
    asyncio.create_task(navigate_page(url.value))


def on_open_progrentis(_):
    asyncio.create_task(navigate_page('https://prod.progrentis.com/signin/tabid/1543/Default.aspx'))


async def inspect_page(_=None):
    try:
        observed = await browser.inspect()
        show_observation(observed)
    except Exception as exc:
        show_message('Inspection error', str(exc), '#b03a2e')


def on_inspect(_):
    asyncio.create_task(inspect_page())


async def ask_agent():
    current_prompt = prompt.value.strip()
    if not current_prompt:
        return
    show_message('You', current_prompt)
    send_button.disabled = True
    stop_button.disabled = False
    try:
        result = await run_agent(current_prompt)
        for item in result.observations:
            show_message('Browser observation', f"{item.get('title', '')} ? {item.get('url', '')}\n{item.get('text', '')[:3000]}", '#426a46')
            show_observation(item)
        show_message('Model interpretation', result.response)
    except asyncio.CancelledError:
        show_message('Agent', 'Run stopped by user.', '#a05a20')
    except Exception as exc:
        show_message('Agent error', str(exc), '#b03a2e')
    finally:
        state['task'] = None
        send_button.disabled = False
        stop_button.disabled = True


def on_send(_):
    if state['task'] is None:
        state['task'] = asyncio.create_task(ask_agent())


def on_stop(_):
    task = state.get('task')
    if task is not None and not task.done():
        task.cancel()


async def sign_in(_=None):
    if not confirmed.value:
        show_message('Sign-in blocked', 'Confirm the official Progrentis origin first.', '#b03a2e')
        return
    settings.PROGRENTIS_USERNAME = username.value
    settings.PROGRENTIS_PASSWORD = password.value
    try:
        result = await browser.authenticate_progrentis('https://prod.progrentis.com')
        show_observation(result['observation'])
        show_message('Sign-in status', result['message'], '#426a46')
        password.value = ''
    except Exception as exc:
        show_message('Sign-in error', str(exc), '#b03a2e')


def on_login(_):
    asyncio.create_task(sign_in())


async def shutdown():
    try:
        await browser.stop()
    finally:
        process = globals().get('model_proc')
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)
        status.value = '<b>Model:</b> stopped ? <b>Browser:</b> stopped'


def on_shutdown(_):
    asyncio.create_task(shutdown())

open_button.on_click(on_open)
progrentis_button.on_click(on_open_progrentis)
inspect_button.on_click(on_inspect)
send_button.on_click(on_send)
stop_button.on_click(on_stop)
login_button.on_click(on_login)
shutdown_button.on_click(on_shutdown)

display(widgets.VBox([
    widgets.HTML('<h2 style="margin-bottom:4px">Progrentis Agent</h2><p>Local Nanbeige ? read-only browser inspection ? ungraded practice help</p>'),
    status,
    widgets.HBox([url, open_button, progrentis_button, inspect_button]),
    page_view,
    widgets.HTML('<h3>Conversation</h3>'), chat_log,
    prompt,
    widgets.HBox([send_button, stop_button]),
    widgets.HTML('<h3>Progrentis sign-in</h3><p>Credentials stay in this kernel and are not sent to Nanbeige. CAPTCHA or ambiguous forms are not bypassed.</p>'),
    widgets.HBox([username, password]), confirmed,
    widgets.HBox([login_button, shutdown_button]),
]))
print('Ready. Model inference is loopback-only; no Cloudflare tunnel is used.')

## Troubleshooting

- **GPU out-of-memory or server exits:** change `MODEL_CTX` from `32768` to `16384` or `8192`; rerun the model start cell. T4 GPU memory is split across devices and the total is not one contiguous pool.
- **Missing model or server:** attach the required Kaggle datasets and verify their folder names / GGUF filename in the config cell.
- **Binary or shared library error:** attach a CUDA-enabled `llama-cpp-bin` compatible with the Kaggle image. The prepare cell verifies `llama-server --version` and options.
- **Long model startup:** the start cell waits up to 15 minutes. Check `/tmp/progrentis-llama-server.log` after a failure.
- **Browser startup:** Playwright Chromium and Linux dependencies are installed in the project setup cell. Rerun that cell if the installation was interrupted.
- **Website unavailable:** this removes the Cloudflare hop between the agent and model only. Kaggle still needs Internet access to reach Progrentis and GitHub.
- **Notebook source is old:** push the latest workspace changes to GitHub `main`, or attach a Kaggle input dataset containing the current project source before starting a new notebook session.